**Transformer-Based English to Tamil Translation (Minimal Encoder-Decoder)**

In this project, I have implemented a complete Transformer architecture from scratch in PyTorch to perform sequence-to-sequence translation from English to Tamil. The model uses:

One custom Encoder Layer and one custom Decoder Layer, both manually built without any reliance on PyTorch's built-in Transformer modules

Reuse of previously implemented components including:

Multi-Head Attention, Position-wise Feed Forward Networks

Residual Connections, Layer Normalization, and Embedding layers

The Decoder additionally incorporates:

Masked Multi-Head Attention to prevent future token leakage

Attention over the encoder output

Text data is processed using torchtext and torchdata for efficient tokenization, batching, and vocabulary management

The model is trained using CrossEntropyLoss and optimized using Stochastic Gradient Descent (SGD)

The translation task is demonstrated on simple English-Tamil sentence pairs, and for development/testing purposes, translations into any target language can be experimented with using tools like Google Translate. This project offers a reproducible, minimalist blueprint of Transformer-based translation systems and helps build an intuitive understanding of attention-based architectures.



In [ ]:
!pip install torchdata==0.6.0 # to be compatible with torch 2.0
!pip install portalocker==2.0.0
!pip install -U torchtext==0.15.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

* Let's import all required libraries

In [ ]:
import torch
from torch import Tensor

import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.nn.functional import one_hot

import torch.optim as optim

#text lib
import torchtext

# tokenizer
from torchtext.data.utils import get_tokenizer

#build vocabulary
from torchtext.vocab import vocab
from torchtext.vocab import build_vocab_from_iterator

# get input_ids (numericalization)
from torchtext.transforms import VocabTransform, LabelToIndex

# get embeddings
from torch.nn import Embedding

from  pprint import pprint
from yaml import safe_load
import copy
import numpy as np
import requests
import math

# Preparing Data

* Source and target text

In [ ]:
src_text = """The most famous ruler of ancient India was Emperor Ashoka.
It was during his period that Buddhism spread to different parts of Asia.
Ashoka gave up war after seeing many people grieving death after the Kalinga war.
He embraced Buddhism and then devoted his life to spread the message of peace and dharma.
His service for the cause of public good was exemplary.
He was the first ruler to give up war after victory.
He was the first to build hospitals for animals.
He was the first to lay roads."""

In [ ]:
tar_text = """பண்டைய இந்திய அரசர்களில் பேரும் புகழும் பெற்ற அரசர் அசோகர் ஆவார்.
இவரது ஆட்சியில் தான் புத்த மதம் ஆசியாவின் பல்வேறு பகுதிகளுக்குப் பரவியது.
கலிங்கப் போருக்குப் பின் பல உயிர்கள் மடிவதைக் கண்டு வருந்தி, போர் தொடுப்பதைக் கைவிட்டார்.
அதற்குப் பிறகு புத்த சமயத்தைத் தழுவி, அமைதியையும் அறத்தையும் பரப்புவதற்காகத் தன் வாழ்வையே அர்ப்பணித்தார்.
பொதுமக்களுக்கு அவர் ஆற்றிய சேவை முன் மாதிரியாக விளங்கியது.
வெற்றிக்குப் பின் போரைத் துறந்த முதல் அரசர் அசோகர்தான்.
உலகிலேயே முதன்முதலாக விலங்குகளுக்கும் தனியே மருத்துவமனை அமைத்துத் தந்தவரும் அசோகரே ஆவார்.
 இன்றும் அவர் உருவாக்கிய சாலைகளை நாம் பயன்படுத்திக்கொண்டு இருக்கிறோம்."""

* Tokenize and build vocabulary using a simple tokenization algorithm

In [ ]:

def seq_len(seq):
  return len(seq.strip('').split(' '))

# check the maximum length of the src and target seq to decide the context length of encdoer and decoder
src_raw_seq = src_text.strip('').split('\n')
src_max_seq_len =max(list(map(seq_len,src_raw_seq)))
print('Source max_seq_length:  ',src_max_seq_len)


tar_raw_seq = tar_text.strip('').split('\n')
tar_max_seq_len =max(list(map(seq_len,tar_raw_seq)))
print('Target max_seq_length: ',tar_max_seq_len)

Source max_seq_length:   16
Target max_seq_length:  11


In [ ]:

class Tokenizer(object):

  def __init__(self,text):
    self.text = text
    self.word_tokenizer = self.word_tokenizer
    self.vocab_size = None
    self.vocab = None

  @staticmethod
  def word_tokenizer(seq):
    return seq.strip('').split(' ')

  def get_tokens(self):
    for sentence in self.text.strip().split('\n'):
      yield self.word_tokenizer(sentence)

  def build_vocab(self):
    self.vocab = build_vocab_from_iterator(self.get_tokens(),
                                  min_freq=1,specials=['<pad>','<start>','<end>','<unk>'])
    self.vocab.set_default_index(self.vocab['<unk>']) # index of OOV
    self.vocab_size = len(self.vocab)
    return self.vocab

  def encode(self,sentence):
    v = self.build_vocab()
    vt = VocabTransform(v)
    token_ids = vt(self.word_tokenizer(sentence))
    # add special tokens
    token_ids.insert(0,v.vocab.get_stoi()['<start>'])
    token_ids.append(v.vocab.get_stoi()['<end>']) # <end>:2
    return torch.tensor(token_ids,dtype=torch.int64)

  def decode(self,ids):
    v = self.build_vocab()
    list_ids = ids.tolist()
    tokens = [v.vocab.get_itos()[id] for id in list_ids]
    return ' '.join(tokens)

  def encode_batch(self,batch_size,max_seq_len):
    batch_data = torch.zeros(size=(batch_size,max_seq_len+2)) # +2 for special tokens
    for i,sentence in enumerate(self.text.strip('').split('\n')):
      token_ids = self.encode(sentence)
      batch_data[i,0:len(token_ids)] = token_ids
    return batch_data.type(dtype=torch.int64)



In [ ]:
batch_size = 8

In [ ]:

src_tokenizer = Tokenizer(src_text)
print(src_tokenizer.encode('The most famous ruler of ancient India was Emperor Ashoka.'))
print(src_tokenizer.encode_batch(batch_size,src_max_seq_len))

tensor([ 1, 27, 49, 39, 15,  8, 28, 24,  5, 22, 20,  2])
tensor([[ 1, 27, 49, 39, 15,  8, 28, 24,  5, 22, 20,  2,  0,  0,  0,  0,  0,  0],
        [ 1, 25,  5, 36, 14, 53, 58, 11, 16,  6, 35, 50,  8, 21,  2,  0,  0,  0],
        [ 1, 19, 40, 17, 18,  9, 56, 47, 52, 43, 32,  9,  4, 26, 61,  2,  0,  0],
        [ 1,  7, 37, 11, 12, 59, 33, 14, 46,  6, 16,  4, 48,  8, 51, 12, 34,  2],
        [ 1, 23, 57, 13,  4, 31,  8, 54, 42,  5, 38,  2,  0,  0,  0,  0,  0,  0],
        [ 1,  7,  5,  4, 10, 15,  6, 41, 17, 18,  9, 60,  2,  0,  0,  0,  0,  0],
        [ 1,  7,  5,  4, 10,  6, 30, 44, 13, 29,  2,  0,  0,  0,  0,  0,  0,  0],
        [ 1,  7,  5,  4, 10,  6, 45, 55,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0]])


In [ ]:
print(src_tokenizer.encode('war war'))

tensor([ 1, 18, 18,  2])


In [ ]:

tar_tokenizer = Tokenizer(tar_text)
print(tar_tokenizer.encode('பண்டைய இந்திய அரசர்களில் பேரும் புகழும் பெற்ற அரசர் அசோகர் ஆவார்.'))
print(tar_tokenizer.encode_batch(batch_size,tar_max_seq_len))

tensor([ 1, 44, 22, 16, 53, 51, 52,  4, 11,  6,  2])
tensor([[ 1, 44, 22, 16, 53, 51, 52,  4, 11,  6,  2,  0,  0],
        [ 1, 25, 20, 39,  8, 59, 19, 49, 43, 47,  2,  0,  0],
        [ 1, 30, 55,  7, 48, 26, 58, 29, 65, 57, 41, 31,  2],
        [ 1, 13, 50,  8, 32, 38, 14, 18, 46, 37, 66, 17,  2],
        [ 1, 54,  5, 21, 34, 64, 61, 68,  2,  0,  0,  0,  0],
        [ 1, 69,  7, 56, 40, 63,  4, 12,  2,  0,  0,  0,  0],
        [ 1, 28, 62, 67, 36, 60, 15, 35, 10,  6,  2,  0,  0],
        [ 1,  9, 23,  5, 27, 33, 42, 45, 24,  2,  0,  0,  0]])


* Let's load the token ids of the words in the sentences of source and target languages

In [ ]:
print(tar_tokenizer.encode('<pad>'))
print(tar_tokenizer.encode('wr dk'))

tensor([1, 0, 2])
tensor([1, 3, 3, 2])


In [ ]:

x = src_tokenizer.encode_batch(batch_size,src_max_seq_len)
y = tar_tokenizer.encode_batch(batch_size,tar_max_seq_len)

In [ ]:
print(y)

tensor([[ 1, 44, 22, 16, 53, 51, 52,  4, 11,  6,  2,  0,  0],
        [ 1, 25, 20, 39,  8, 59, 19, 49, 43, 47,  2,  0,  0],
        [ 1, 30, 55,  7, 48, 26, 58, 29, 65, 57, 41, 31,  2],
        [ 1, 13, 50,  8, 32, 38, 14, 18, 46, 37, 66, 17,  2],
        [ 1, 54,  5, 21, 34, 64, 61, 68,  2,  0,  0,  0,  0],
        [ 1, 69,  7, 56, 40, 63,  4, 12,  2,  0,  0,  0,  0],
        [ 1, 28, 62, 67, 36, 60, 15, 35, 10,  6,  2,  0,  0],
        [ 1,  9, 23,  5, 27, 33, 42, 45, 24,  2,  0,  0,  0]])


In [ ]:
label =(y!=0)

* Define the context lengths for encoder and decoder

In [ ]:

enc_ctxt_len = src_max_seq_len+2
dec_ctxt_len = tar_max_seq_len+2

# Load configuration file

In [ ]:

config_url = "https://raw.githubusercontent.com/Arunprakash-A/LLM-from-scratch-PyTorch/main/config_files/enc_config.yml"
response = requests.get(config_url)
config = response.content.decode("utf-8")
config = safe_load(config)
pprint(config)

{'input': {'batch_size': 10, 'embed_dim': 32, 'seq_len': 8, 'vocab_size': 10},
 'model': {'d_ff': 128,
           'd_model': 32,
           'dk': 4,
           'dq': 4,
           'dv': 4,
           'n_heads': 8,
           'n_layers': 6}}


In [ ]:

src_vocab_size =src_tokenizer.vocab_size
batch_size = x.shape[0]
embed_dim = config['input']['embed_dim']

In [ ]:

dq = torch.tensor(config['model']['dq'])
dk = torch.tensor(config['model']['dk'])
dv = torch.tensor(config['model']['dv'])
dmodel = embed_dim
heads = torch.tensor(config['model']['n_heads'])
d_ff = config['model']['d_ff']

In [ ]:

config_url = "https://raw.githubusercontent.com/Arunprakash-A/LLM-from-scratch-PyTorch/main/config_files/dec_config.yml"
response = requests.get(config_url)
config = response.content.decode("utf-8")
config = safe_load(config)
pprint(config)

{'input': {'batch_size': 10, 'embed_dim': 32, 'seq_len': 8, 'vocab_size': 12},
 'model': {'d_ff': 128,
           'd_model': 32,
           'dk': 4,
           'dq': 4,
           'dv': 4,
           'n_heads': 8,
           'n_layers': 6}}


In [ ]:

tar_vocab_size = tar_tokenizer.vocab_size

In [ ]:
print(tar_vocab_size)

70


# Encoder


In [ ]:
class MHA(nn.Module):
   def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHA,self).__init__()
    self.num_heads = heads
    self.d_model = dmodel
    self.dq = dq
    self.dk = dk
    self.dv = dv

    # Linear layers for queries, keys, and values
    torch.manual_seed(43)
    self.W_q = nn.Linear(dmodel, dq * heads)
    torch.manual_seed(44)
    self.W_k = nn.Linear(dmodel, dk * heads)
    torch.manual_seed(45)
    self.W_v = nn.Linear(dmodel, dv * heads)
    # Linear layer for output
    torch.manual_seed(46)
    self.W_o = nn.Linear(heads * dv, dmodel)
    # self.initialize_weights()


   def initialize_weights(self):
        # Initialize W_Q with seed 43
        torch.manual_seed(43)
        self.W_q.weight=nn.Parameter(torch.randn(dq*heads,dmodel))
        self.W_q.bias = nn.Parameter(torch.randn(dmodel))

        # Initialize W_K with seed 44
        torch.manual_seed(44)
        self.W_k.weight=nn.Parameter(torch.randn(dk*heads,dmodel))
        self.W_k.bias = nn.Parameter(torch.randn(dmodel))

        # Initialize W_V with seed 45
        torch.manual_seed(45)
        self.W_v.weight=nn.Parameter(torch.randn(dv*heads,dmodel))
        self.W_v.bias = nn.Parameter(torch.randn(dmodel))

        # Initialize W_O with seed 46
        torch.manual_seed(46)
        self.W_o.weight=nn.Parameter(torch.randn(dmodel,dv*heads))
        self.W_o.bias = nn.Parameter(torch.randn(dmodel))
   def forward(self,H=None):
    '''
    Input: Size [BSxTxdmodel]
    Output: Size[BSxTxdmodel]
    '''
    batch_size, seq_len, dmodel = H.size()

    # Linear transformations
    Q = self.W_q(H).view(batch_size, seq_len, self.num_heads, self.dq).transpose(1, 2)  # [BS x heads x T x dq]
    K = self.W_k(H).view(batch_size, seq_len, self.num_heads, self.dk).transpose(1, 2)  # [BS x heads x T x dk]
    V = self.W_v(H).view(batch_size, seq_len, self.num_heads, self.dv).transpose(1, 2)  # [BS x heads x T x dv]

    # Scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.dk ** 0.5)  # [BS x heads x T x T]
    attn_weights = F.softmax(scores, dim=-1)  # [BS x heads x T x T]
    output = torch.matmul(attn_weights, V)  # [BS x heads x T x dv]

    # Concatenate heads and apply output linear transformation
    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.num_heads * self.dv)  # [BS x T x (heads * dv)]
    out = self.W_o(output)  # [BS x T x d_model]

    return out
class FFN(nn.Module):
  def __init__(self,dmodel,d_ff,layer=0):
    super(FFN,self).__init__()
    # First linear layer to expand the dimension
    torch.manual_seed(47)
    self.fc1 = nn.Linear(dmodel, d_ff)
    # Second linear layer to project it back to the original dimension
    torch.manual_seed(48)
    self.fc2 = nn.Linear(d_ff, dmodel)
    # ReLU activation
    self.relu = nn.ReLU()
    # self.initialize_weights()


  def initialize_weights(self):
        # Initialize fc1 with seed 47
        torch.manual_seed(47)
        self.fc1.weight = nn.Parameter(torch.randn(d_ff, dmodel))  # d_ff as output features, d_model as input features
        self.fc1.bias = nn.Parameter(torch.randn(d_ff))

        # Initialize fc2 with seed 48
        torch.manual_seed(48)
        self.fc2.weight = nn.Parameter(torch.randn(dmodel, d_ff))
        self.fc2.bias = nn.Parameter(torch.randn(dmodel))
  def forward(self,x):
    '''
    input: size [BSxTxdmodel]
    output: size [BSxTxdmodel]
    '''
    return self.fc2(self.relu(self.fc1(x)))

    return out
class Prediction(nn.Module):
  def __init__(self,dmodel,vocab_size):
    super(Prediction,self).__init__()
    torch.manual_seed(49)
    self.linear = nn.Linear(dmodel, vocab_size)

  def forward(self,representations):
    '''
    input: size [bsxTxdmodel]
    output: size [bsxTxvocab_size]
    '''
    out = self.linear(representations)
    return out

class EncoderLayer(nn.Module):
  def __init__(self,dmodel,dq,dk,dv,d_ff,heads):
    super(EncoderLayer,self).__init__()
    self.mha = MHA(dmodel,dq,dk,dv,heads)
    self.layer_norm_mha = torch.nn.LayerNorm(dmodel)
    self.layer_norm_ffn = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,x):

    # do a forward pass
    output=self.mha(x)
    output=self.layer_norm_mha(output)
    output=self.ffn(output)
    out=self.layer_norm_ffn(output)

    return out

class Encoder(nn.Module):
  def __init__(self,vocab_size,embed_dim,dq,dk,dv,d_ff,heads,num_layers=1):
    super(Encoder,self).__init__()
    self.layers = nn.ModuleList([EncoderLayer(embed_dim, dq, dk, dv, d_ff, heads) for _ in range(num_layers)])  # List of Encoder layers


  def forward(self,x):
    '''
    The input should be tokens ids of size [BS,T]
    '''
    out=x
    for layer in self.layers:
            out = layer(out)  # Pass the embeddings through the encoder layers, size: [BS, T, d_model]

    return out


# Decoder

In [ ]:
class MHCA(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHCA,self).__init__()
    self.heads = heads
    self.dk = dk
    self.dv = dv
    # Linear layers for queries, keys, and values
    torch.manual_seed(43)
    self.linear_q = nn.Linear(dmodel, dq * heads)
    torch.manual_seed(44)
    self.linear_k = nn.Linear(dmodel, dk * heads)
    torch.manual_seed(45)
    self.linear_v = nn.Linear(dmodel, dv * heads)
    # Linear layer for output
    torch.manual_seed(46)
    self.linear_out = nn.Linear(heads * dv, dmodel)
    # self.initialize_weights()


  def initialize_weights(self):
      # Initialize W_Q with seed 43
      torch.manual_seed(43)
      self.linear_q.weight=nn.Parameter(torch.randn(dq*heads,dmodel))
      self.linear_q.bias = nn.Parameter(torch.randn(dmodel))

      # Initialize W_K with seed 44
      torch.manual_seed(44)
      self.linear_k.weight=nn.Parameter(torch.randn(dk*heads,dmodel))
      self.linear_k.bias = nn.Parameter(torch.randn(dmodel))

      # Initialize W_V with seed 45
      torch.manual_seed(45)
      self.linear_v.weight=nn.Parameter(torch.randn(dv*heads,dmodel))
      self.linear_v.bias = nn.Parameter(torch.randn(dmodel))

      # Initialize W_O with seed 46
      torch.manual_seed(46)
      self.linear_out.weight=nn.Parameter(torch.randn(dmodel,dv*heads))
      self.linear_out.bias = nn.Parameter(torch.randn(dmodel))


  def forward(self, queries, keys, values, mask=None):
    batch_size = queries.size(0)
    seq_len_q = queries.size(1)  # Length of query sequence
    seq_len_k = keys.size(1)      # Length of key sequence

    # Linear projections
    Q = self.linear_q(queries)  # Shape: (batch_size, seq_len_q, heads * dq)
    K = self.linear_k(keys)      # Shape: (batch_size, seq_len_k, heads * dk)
    V = self.linear_v(values)    # Shape: (batch_size, seq_len_k, heads * dv)

    # Reshape to (batch_size, seq_len, heads, dim_per_head)
    Q = Q.view(batch_size, seq_len_q, self.heads, self.dk).transpose(1, 2)  # Shape: (batch_size, heads, seq_len_q, dk)
    K = K.view(batch_size, seq_len_k, self.heads, self.dk).transpose(1, 2)  # Shape: (batch_size, heads, seq_len_k, dk)
    V = V.view(batch_size, seq_len_k, self.heads, self.dv).transpose(1, 2)  # Shape: (batch_size, heads, seq_len_k, dv)

    # Scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.dk ** 0.5)  # Shape: (batch_size, heads, seq_len_q, seq_len_k)

    # if mask is not None:
    #     scores = scores.masked_fill(mask == 0, float('-inf'))  # Apply mask if provided

    attention_weights = F.softmax(scores, dim=-1)  # Shape: (batch_size, heads, seq_len_q, seq_len_k)

    out = torch.matmul(attention_weights, V)  # Shape: (batch_size, heads, seq_len_q, dv)

    # Concatenate heads and put through final linear layer
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len_q, -1)  # Shape: (batch_size, seq_len_q, heads * dv)
    out = self.linear_out(out)  # Shape: (batch_size, seq_len_q, d_model)

    return out

class MHMA(nn.Module):
    def __init__(self, d_model, dq, dk, dv, heads, mask=None):
        super(MHMA, self).__init__()

        self.heads = heads
        self.dk = dk
        self.dv = dv
        self.dq = dq

        # Linear layers for queries, keys, and values
        torch.manual_seed(43)
        self.linear_q = nn.Linear(dmodel, dq * heads)
        torch.manual_seed(44)
        self.linear_k = nn.Linear(dmodel, dk * heads)
        torch.manual_seed(45)
        self.linear_v = nn.Linear(dmodel, dv * heads)
        # Linear layer for output
        torch.manual_seed(46)
        self.linear_out = nn.Linear(heads * dv, dmodel)
        # self.initialize_weights()



    def initialize_weights(self):
          # Initialize W_Q with seed 43
          torch.manual_seed(43)
          self.linear_q.weight=nn.Parameter(torch.randn(dq*heads,dmodel))
          self.linear_q.bias = nn.Parameter(torch.randn(dmodel))

          # Initialize W_K with seed 44
          torch.manual_seed(44)
          self.linear_k.weight=nn.Parameter(torch.randn(dk*heads,dmodel))
          self.linear_k.bias = nn.Parameter(torch.randn(dmodel))

          # Initialize W_V with seed 45
          torch.manual_seed(45)
          self.linear_v.weight=nn.Parameter(torch.randn(dv*heads,dmodel))
          # self.linear_v.bias = nn.Parameter(torch.randn(dmodel))

          # Initialize W_O with seed 46
          torch.manual_seed(46)
          self.linear_out.weight=nn.Parameter(torch.randn(dmodel,dv*heads))
          # self.linear_out.bias = nn.Parameter(torch.randn(dmodel))

    def forward(self, queries, keys, values):
        batch_size = queries.size(0)
        seq_len_q = queries.size(1)
        seq_len_k = keys.size(1)

        # Linear projections for queries, keys, and values
        Q = self.linear_q(queries)  # Shape: (batch_size, seq_len_q, heads * dq)
        K = self.linear_k(keys)      # Shape: (batch_size, seq_len_k, heads * dk)
        V = self.linear_v(values)    # Shape: (batch_size, seq_len_k, heads * dv)

        # Reshape to (batch_size, heads, seq_len, dim_per_head)
        Q = Q.view(batch_size, seq_len_q, self.heads, self.dq).transpose(1, 2)
        K = K.view(batch_size, seq_len_k, self.heads, self.dk).transpose(1, 2)
        V = V.view(batch_size, seq_len_k, self.heads, self.dv).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.dk ** 0.5)  # Shape: (batch_size, heads, seq_len_q, seq_len_k)
        mask = torch.tril(torch.ones(seq_len_q, seq_len_k)).type(torch.int32)
        mask = mask.unsqueeze(0).unsqueeze(0)  # Add batch and heads dimensions
        mask = mask.expand(batch_size, heads, -1, -1)  # (batch_size, heads, seq_len, seq_len)

        # Apply the mask
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # Softmax over the last dimension (keys)
        attention_weights = F.softmax(scores, dim=-1)  # Shape: (batch_size, heads, seq_len_q, seq_len_k)

        # Attention output
        out = torch.matmul(attention_weights, V)  # Shape: (batch_size, heads, seq_len_q, dv)

        # Concatenate heads and project back to d_model
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len_q, -1)  # Shape: (batch_size, seq_len_q, heads * dv)
        out = self.linear_out(out)  # Shape: (batch_size, seq_len_q, d_model)

        return out
class DecoderLayer(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,d_ff,heads,mask=None):
    super(DecoderLayer,self).__init__()
    self.mhma = MHMA(dmodel,dq,dk,dv,heads,mask)
    self.mhca = MHCA(dmodel,dq,dk,dv,heads)
    self.layer_norm_mhma = torch.nn.LayerNorm(dmodel)
    self.layer_norm_mhca = torch.nn.LayerNorm(dmodel)
    self.layer_norm_ffn = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,embedded_input,enc_rep):
    mhma_out = self.mhma(embedded_input, embedded_input, embedded_input)
    mhma_out = self.layer_norm_mhma(mhma_out + embedded_input)  # Residual connection

    # Multi-head cross-attention with residual connection
    mhca_out = self.mhca(mhma_out, enc_rep, enc_rep)
    mhca_out = self.layer_norm_mhca(mhca_out + mhma_out)  # Residual connection

    # Feed-forward network with residual connection
    ffn_out = self.ffn(mhca_out)
    out = self.layer_norm_ffn(ffn_out + mhca_out)  # Residual connection


    return out

class Decoder(nn.Module):
  def __init__(self,vocab_size,dmodel,dq,dk,dv,d_ff,heads,mask,num_layers=1):
    super(Decoder,self).__init__()
    self.layer = DecoderLayer(dmodel, dq, dk, dv, d_ff, heads,mask)
    self.out = Prediction(dmodel, vocab_size)  # Output layer

  def forward(self,in_token_ids,enc_rep):
    # print(in_token_ids.shape,"tar shape")
    out = self.layer(in_token_ids,enc_rep)  # Pass the embeddings through the dncoder layers, size: [BS, T, d_model]
    out =  self.out(out)


    return out

# Positional Embedding


In [ ]:


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create a matrix of (max_len, d_model) with positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))  # (d_model/2,)

        # Compute the sine and cosine for each position and dimension
        pe[:, 0::2] = torch.sin(position * div_term)  # Apply sine to even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Apply cosine to odd indices

        pe = pe.unsqueeze(0)  # Add a batch dimension (1, max_len, d_model)
        self.register_buffer('pe', pe)  # Register pe as a persistent buffer

    def forward(self, x):
        x=x.view(1,x.shape[0],x.shape[1])
        x = x + self.pe[:, :x.size(1)]  # x.size(1) is the length of the sequence
        return self.dropout(x)


# Generate target mask

  * We will be passing the causal mask for the decoder layer as one of its arguments

In [ ]:
mask = (torch.triu(torch.ones(dec_ctxt_len,dec_ctxt_len)) == 1).transpose(0,1)
mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
print(mask)

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


# Transformer

In [ ]:
class Transformer(nn.Module):

  def __init__(self,src_vocab_size,tar_vocab_szie,src_seq_len,tar_seq_len,dmodel,dq,dk,dv,d_ff,heads,target_mask,num_layers=1):
    super(Transformer,self).__init__()
    self.src_embeddings = nn.Embedding(src_vocab_size,embed_dim)
    self.tar_embeddings = nn.Embedding(tar_vocab_size,embed_dim)
    self.pos_embeddings = PositionalEncoding(dmodel)
    self.encoder = Encoder(src_vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers)
    self.decoder = Decoder(tar_vocab_size,dmodel,dq,dk,dv,d_ff,heads,target_mask,num_layers)

  def forward(self,src_token_ids,in_token_ids):
    out = self.encoder(self.pos_embeddings(self.src_embeddings(src_token_ids)))
    out = self.decoder(self.tar_embeddings(in_token_ids),out)
    return out

In [ ]:
model = Transformer(src_vocab_size,tar_vocab_size,enc_ctxt_len,dec_ctxt_len,dmodel,dq,dk,dv,d_ff,heads,mask)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
def train(src_token_ids,tgt_token_ids,labels,epochs=1000):
  loss_trace = []
  for epoch in range(epochs):
        model.train()
        total_loss = 0

        # Go through each sequence in the batch
        for i in range(len(src_token_ids)):
            src_seq = src_token_ids[i]  # Encoder input for the current sequence
            tgt_seq = tgt_token_ids[i]  # Target sequence for the current sequence

            # Initialize the decoder input with the <start> token and paddings
            decoder_input = torch.tensor([1] + [0] * (dec_ctxt_len - 1), dtype=torch.long)  # 1 = <start> token

            optimizer.zero_grad()
            total_loss_seq=0

            # Autoregressive training loop
            for t in range(1, len(tgt_seq)):
                # Get the model's prediction for the current timestep
                output = model(src_seq, decoder_input.unsqueeze(0))  # Shape: (1, context_length, vocab_size)
                # print(output.shape,"output shape") #torch.Size([1, 13, 70]) output shape
                # print(output[0,t-1].shape,"output [0,t-1] shape") # torch.Size([70]) output [0,t-1] shape

                # Extract only the last output token (to predict the t-th token)
                output_token = output[0, t - 1].unsqueeze(0)  # Shape: (1*vocab_size,)

                # Calculate loss only on the predicted token
                # print(output_token.shape,"output_token shape") #torch.Size([1, 70]) output_token shape
                # print(tgt_seq[t].unsqueeze(0).shape) #torch.Size([1])
                loss = criterion(output_token, tgt_seq[t].unsqueeze(0))
                total_loss_seq+=loss.item()
                loss.backward(retain_graph=True)

                # Teacher forcing: add the correct token to decoder input for the next prediction
                decoder_input[t] = tgt_seq[t]

            optimizer.step()
            temp_loss=total_loss_seq/len(tgt_seq)
            total_loss+=temp_loss
        loss_trace.append(total_loss)
        print(f'epoch {epoch+1} loss:',total_loss)
  return loss_trace

In [ ]:
loss_trace=train(x,y,label,1000)

epoch 1 loss: 30.163303315639496
epoch 2 loss: 25.885372692575825
epoch 3 loss: 23.45606471827397
epoch 4 loss: 21.17292339813251
epoch 5 loss: 18.892002109724743
epoch 6 loss: 16.639385043428494
epoch 7 loss: 14.476850505058582
epoch 8 loss: 12.368476749182893
epoch 9 loss: 10.668220977657116
epoch 10 loss: 8.877256887248503
epoch 11 loss: 7.2846889747306705
epoch 12 loss: 6.041004564362362
epoch 13 loss: 5.068267524457322
epoch 14 loss: 4.367584725555319
epoch 15 loss: 3.8119416130133543
epoch 16 loss: 3.4004873790097636
epoch 17 loss: 3.061066207655061
epoch 18 loss: 2.795319168076206
epoch 19 loss: 2.6217419252766723
epoch 20 loss: 2.3577289037191527
epoch 21 loss: 2.204540583394611
epoch 22 loss: 2.055381975763549
epoch 23 loss: 1.9154368894747817
epoch 24 loss: 1.7650048667654539
epoch 25 loss: 1.6946355912189643
epoch 26 loss: 1.5311198711144522
epoch 27 loss: 1.4816567508342604
epoch 28 loss: 1.4673608228062778
epoch 29 loss: 1.6236001251605698
epoch 30 loss: 1.690875427354163


## Run the model AutoRegressively

In [ ]:
@torch.inference_mode()
def inference(test_input):
  '''
  Run the model in autoregressive fashion and store the output at each time step in a list
  '''
  output_tokenid=[1]
  src_seq = test_input  # Encoder input for the current sequence

  # Initialize the decoder input with the <start> token and paddings
  decoder_input = torch.tensor([1] + [0] * (dec_ctxt_len - 1), dtype=torch.long)  # 1 = <start> token

  # Autoregressive training loop
  for t in range(1, dec_ctxt_len):
      output = model(src_seq, decoder_input.unsqueeze(0))  # Shape: (1, context_length, vocab_size)

      # Extract only the last output token (to predict the t-th token)
      output_token = output[0, t - 1].unsqueeze(0)  # Shape: (1*vocab_size,)
      token_id=torch.argmax(output_token.squeeze(0))
      output_tokenid.append(token_id)
      decoder_input[t] = token_id

  return torch.tensor(output_tokenid)

In [ ]:
for token_ids in x:
  print(src_tokenizer.decode(token_ids))
  print(tar_tokenizer.decode(inference(token_ids)))

<start> The most famous ruler of ancient India was Emperor Ashoka. <end> <pad> <pad> <pad> <pad> <pad> <pad>
<start> பண்டைய இந்திய அரசர்களில் பேரும் புகழும் பெற்ற அரசர் அசோகர் ஆவார். <end> <pad> <pad>
<start> It was during his period that Buddhism spread to different parts of Asia. <end> <pad> <pad> <pad>
<start> இவரது ஆட்சியில் தான் புத்த மதம் ஆசியாவின் பல்வேறு பகுதிகளுக்குப் பரவியது. <end> <pad> <pad>
<start> Ashoka gave up war after seeing many people grieving death after the Kalinga war. <end> <pad> <pad>
<start> கலிங்கப் போருக்குப் பின் பல உயிர்கள் மடிவதைக் கண்டு வருந்தி, போர் தொடுப்பதைக் கைவிட்டார். <end>
<start> He embraced Buddhism and then devoted his life to spread the message of peace and dharma. <end>
<start> அதற்குப் பிறகு புத்த சமயத்தைத் தழுவி, அமைதியையும் அறத்தையும் பரப்புவதற்காகத் தன் வாழ்வையே அர்ப்பணித்தார். <end>
<start> His service for the cause of public good was exemplary. <end> <pad> <pad> <pad> <pad> <pad> <pad>
<start> பொதுமக்களுக்கு அவர் ஆற்றிய சேவை முன் மாதிரி